In [0]:
from pyspark.sql import functions as F

# Load source tables
feedback = spark.table("cpx_dataanalytics_gold.marketing.mkt_adobe_message_feedback")
tracking = spark.table("cpx_dataanalytics_gold.marketing.mkt_adobe_email_tracking")
email_addr = spark.table("cpx_dataanalytics_gold.marketing.cpx_d_customer_email_address")

# Email audience: sent + opened before July 31
df_send_email = (
    feedback.alias("F")
    .join(tracking.alias("T"), F.col("F.message_profile_id") == F.col("T.message_profile_id"), "inner")
    .join(email_addr.alias("E"), F.col("E.Email_Address") == F.col("F.Email_Address"), "left")
    .filter(
        (F.col("F.journey_id") == "96b494ce-f57e-4068-b24c-049fd5684fee")
        & F.col("F.journey_action_id").isin(
            "ef8134cd-3dd9-4553-b8cf-c273db9d84b9",
            "c6ba2ed4-5814-417d-8d46-281932defbc5"
        )
        & (F.col("F.feedback_status") == "sent")
        & (F.col("T.interaction_type") == "open")
        & (F.to_date("T.timestamp") <= "2026-07-31")
    )
    .select(
        F.col("F.Email_Address"),
        F.col("E.CDE_ID"),
        F.col("F.journey_id").alias("Journey_ID")
    )
    .distinct()
)

In [0]:
# Ticket transactions for Minions & Monsters in July 2026
transaction = spark.table("cpx_dataanalytics_gold.customer360.c360_f_transaction")
film = spark.table("cpx_dataanalytics_gold.customer360.cpx_d_film")
ticket_type = spark.table("cpx_dataanalytics_gold.customer360.cpx_d_ticket_type")

df_transaction = (
    transaction.alias("T1")
    .join(film.alias("T3"), F.col("T3.film_id") == F.col("T1.film_id"), "inner")
    .join(ticket_type.alias("T6"), F.col("T6.tickettypeskey") == F.col("T1.TicketTypeSkey"), "left")
    .filter(
        (F.col("T1.TicketFLAG") == "Yes")
        & (F.col("T1.SCENEFLAG") == "Yes")
        & (F.col("T6.paid_attendance_inflag") == "Yes")
        & F.col("T1.TransactionDateid").between(20260701, 20260731)
        & F.col("T1.CDE_ID").isNotNull()
        & (F.col("T3.title_name") == "Minions & Monsters")
    )
    .groupBy(
        F.col("T1.CDE_ID"),
        F.col("T1.Transaction_Number"),
        F.col("T1.LocationId"),
        F.to_date(F.col("T1.VisitDateId").cast("string"), "yyyyMMdd").alias("Visit_Date")
    )
    .agg(
        F.sum("T1.Quantity").alias("No_of_tickets"),
        F.sum("T1.Net_Revenue").alias("Net_Revenue_Tickets")
    )
)


In [0]:
# Point redemptions in July 2026
df_redemption = (
    transaction.alias("T1")
    .filter(
        (F.col("T1.SCENE_Points_Redeemed") > 0)
        & F.col("T1.TransactionDateid").between(20260701, 20260731)
        & F.col("T1.CDE_ID").isNotNull()
    )
    .groupBy(
        F.col("T1.CDE_ID"),
        F.col("T1.Transaction_Number"),
        F.col("T1.LocationId")
    )
    .agg(
        F.sum("T1.SCENE_Points_Redeemed").alias("Points_redeemed")
    )
)

In [0]:
# Join transactions with redemptions, determine eligibility
df_txn_table1 = (
    df_transaction.alias("T")
    .join(
        df_redemption.alias("R"),
        (F.col("R.Transaction_Number") == F.col("T.Transaction_Number"))
        & (F.col("R.CDE_ID") == F.col("T.CDE_ID"))
        & (F.col("R.LocationId") == F.col("T.LocationId")),
        "left"
    )
    .filter(F.col("T.No_of_tickets") > 0)
    .groupBy(
        F.col("T.CDE_ID"),
        F.col("T.Visit_Date"),
        F.col("T.Transaction_Number")
    )
    .agg(
        F.sum("T.No_of_tickets").alias("No_of_tickets"),
        F.sum("T.Net_Revenue_Tickets").alias("Net_Revenue_Tickets"),
        F.sum("R.Points_redeemed").alias("Points_redeemed")
    )
)

df_txn_table = df_txn_table1.withColumn(
    "Eligibility",
    F.when(F.col("Points_redeemed") > 0, "Not Eligible").otherwise("Eligible")
)

# Active Scene+ membership lookup
df_scene = (
    spark.table("cpx_dataanalytics_gold.customer360.cpx_d_customer")
    .filter(F.col("customer_status") == "Active")
    .groupBy("cde_id")
    .agg(F.max("scene_membership_id").alias("scene_membership_id"))
)

# Final result: eligible customers from email audience with visit date breakdown
df_result = (
    df_send_email.alias("E")
    .join(
        df_txn_table.alias("TT"),
        F.col("TT.CDE_ID") == F.col("E.CDE_ID"),
        "inner"
    )
    .join(
        df_scene.alias("S"),
        F.col("S.cde_id") == F.col("E.CDE_ID"),
        "left"
    )
    .filter(F.col("TT.Eligibility") == "Eligible")
    .groupBy(
        F.col("E.CDE_ID"),
        F.col("S.scene_membership_id"),
        F.col("TT.Visit_Date")
    )
    .agg(
        F.sum("TT.No_of_tickets").alias("No_of_tickets")
    )
)



In [0]:
# Row and CDE_ID counts
display(
    df_result.select(
        F.count("*").alias("total_rows"),
        F.countDistinct("CDE_ID").alias("distinct_cde_ids")
    )
)

In [0]:
# ── How many eligible members visited an LBE within 3 weeks of their Minions purchase? ──
location = spark.table("cpx_dataanalytics_gold.customer360.cpx_d_location")
date_dim = spark.table("cpx_dataanalytics_gold.customer360.cpx_d_date")

lbe_locations = F.broadcast(
    location.filter(F.col("LC_Location_Type_Description") == "Restaurant/Store/Games")
    .select("LocationID", "LC_Trade_Brand_Description")
)

df_purchase_dates = F.broadcast(df_result.select("CDE_ID", "Visit_Date").distinct())

df_lbe_txns = (
    transaction.alias("T")
    .filter(
        (F.col("T.SCENEFLAG") == "Yes")
        & (F.col("T.CDE_ID").isNotNull())
        & (F.col("T.VisitDateId").between("20260701", "20260821  "))
    )
    .join(lbe_locations.alias("L"), F.col("L.LocationID") == F.col("T.LocationId"), "inner")
    .join(date_dim.alias("D"), F.col("D.dateid") == F.col("T.VisitDateId"), "inner")
    .select(
        F.col("T.CDE_ID"),
        F.col("D.datefull").alias("LBE_Visit_Date"),
        F.col("T.LocationId"),
        F.col("T.Transaction_Number"),
        F.col("L.LC_Trade_Brand_Description").alias("LBE_Brand"),
        F.col("T.Net_Revenue"),
        F.col("T.Quantity")
    )
)

df_cross_visit = (
    df_purchase_dates.alias("P")
    .join(
        df_lbe_txns.alias("LBE"),
        (F.col("LBE.CDE_ID") == F.col("P.CDE_ID"))
        & (F.col("LBE.LBE_Visit_Date") >= F.col("P.Visit_Date"))
        & (F.col("LBE.LBE_Visit_Date") <= F.date_add(F.col("P.Visit_Date"), 21)),
        "inner"
    )
    .select(
        F.col("P.CDE_ID"),
        F.col("P.Visit_Date").alias("Minions_Purchase_Date"),
        F.col("LBE.LBE_Visit_Date"),
        F.col("LBE.Transaction_Number"),
        F.col("LBE.LBE_Brand"),
        F.col("LBE.Net_Revenue")
    )
)

# Summary stats — now broken out by LBE_Brand
_cross_deduped = df_cross_visit.drop("Minions_Purchase_Date").distinct()
df_cross_summary = (
    _cross_deduped
    .groupBy("LBE_Brand")
    .agg(
        F.countDistinct("CDE_ID").alias("Members_Who_Visited_LBE"),
        F.count("*").alias("Total_LBE_Transactions"),
        F.round(F.sum("Net_Revenue"), 2).alias("Total_LBE_Revenue"),
        F.round(F.sum("Net_Revenue") / F.countDistinct("CDE_ID"), 2).alias("Avg_Revenue_Per_Member")
    )
    .orderBy("LBE_Brand")
)

In [0]:
# # ── How many eligible members visited an LBE within 3 weeks of their Minions purchase? ──
# location = spark.table("cpx_dataanalytics_gold.customer360.cpx_d_location")
# date_dim = spark.table("cpx_dataanalytics_gold.customer360.cpx_d_date")

# # Cache small LBE location lookup (broadcast-friendly)
# lbe_locations = F.broadcast(
#     location.filter(F.col("LC_Location_Type_Description") == "Restaurant/Store/Games")
#     .select("LocationID", "LC_Trade_Brand_Description")
# )

# # Get purchase dates per member from df_result
# df_purchase_dates = F.broadcast(df_result.select("CDE_ID", "Visit_Date").distinct())

# # LBE transactions - push VisitDateId predicate BEFORE join to reduce scan
# df_lbe_txns = (
#     transaction.alias("T")
#     .filter(
#         (F.col("T.SCENEFLAG") == "Yes")
#         & (F.col("T.CDE_ID").isNotNull())
#         & (F.col("T.VisitDateId").between("20260701", "20260821"))
#     )
#     .join(lbe_locations.alias("L"), F.col("L.LocationID") == F.col("T.LocationId"), "inner")
#     .join(date_dim.alias("D"), F.col("D.dateid") == F.col("T.VisitDateId"), "inner")
#     .select(
#         F.col("T.CDE_ID"),
#         F.col("D.datefull").alias("LBE_Visit_Date"),
#         F.col("T.LocationId"),
#         F.col("T.Transaction_Number"),
#         F.col("L.LC_Trade_Brand_Description").alias("LBE_Brand"),
#         F.col("T.Net_Revenue"),
#         F.col("T.Quantity")
#     )
# )

# # Join: LBE visit must be within 21 days AFTER Minions purchase (broadcast the small side)
# df_cross_visit = (
#     df_purchase_dates.alias("P")
#     .join(
#         df_lbe_txns.alias("LBE"),
#         (F.col("LBE.CDE_ID") == F.col("P.CDE_ID"))
#         & (F.col("LBE.LBE_Visit_Date") >= F.col("P.Visit_Date"))
#         & (F.col("LBE.LBE_Visit_Date") <= F.date_add(F.col("P.Visit_Date"), 21)),
#         "inner"
#     )
#     .select(
#         F.col("P.CDE_ID"),
#         F.col("P.Visit_Date").alias("Minions_Purchase_Date"),
#         F.col("LBE.LBE_Visit_Date"),
#         F.col("LBE.Transaction_Number"),
#         F.col("LBE.LBE_Brand"),
#         F.col("LBE.Net_Revenue")
#     )
# )

# # Summary stats — deduplicate to prevent fan-out from multiple purchase dates
# _cross_deduped = df_cross_visit.drop("Minions_Purchase_Date").distinct()
# df_cross_summary = _cross_deduped.agg(
#     F.countDistinct("CDE_ID").alias("Members_Who_Visited_LBE"),
#     F.count("*").alias("Total_LBE_Transactions"),
#     F.round(F.sum("Net_Revenue"), 2).alias("Total_LBE_Revenue"),
#     F.round(F.sum("Net_Revenue") / F.countDistinct("CDE_ID"), 2).alias("Avg_Revenue_Per_Member")
# )



In [0]:
display(df_cross_summary)

In [0]:
# ── P12M LBE visit history (Jul 2025 – Jun 2026) for cross-visitors ──────────
# Reuse lbe_locations from Cell 8; broadcast the small member list
eligible_ids = F.broadcast(df_cross_visit.select("CDE_ID").distinct())

# Push VisitDateId predicate + member filter before expensive joins
df_p12m_lbe = (
    transaction.alias("T")
    .filter(
        (F.col("T.SCENEFLAG") == "Yes")
        & (F.col("T.CDE_ID").isNotNull())
        & (F.col("T.VisitDateId").between("20250701", "20260630"))
    )
    .join(eligible_ids.alias("EL"), F.col("EL.CDE_ID") == F.col("T.CDE_ID"), "inner")
    .join(lbe_locations.alias("L"), F.col("L.LocationID") == F.col("T.LocationId"), "inner")
    .join(date_dim.alias("D"), F.col("D.dateid") == F.col("T.VisitDateId"), "inner")
    .select(
        F.col("T.CDE_ID"),
        F.col("T.Transaction_Number"),
        F.col("T.LocationId"),
        F.col("L.LC_Trade_Brand_Description").alias("LBE_Brand"),
        F.col("D.datefull").alias("Visit_Date"),
        F.col("T.TicketFLAG"),
        F.col("T.ConcessionFLAG"),
        F.col("T.Net_Revenue"),
        F.col("T.Quantity")
    )
)

In [0]:
# ── Aggregate P12M LBE behavior per member ────────────────────────────────────
from pyspark.sql.window import Window

# Distinct visits per member (deduplicate by visit date + location)
df_lbe_visits = (
    df_p12m_lbe
    .select("CDE_ID", "Visit_Date", "LocationId", "LBE_Brand")
    .distinct()
)

df_member_lbe_summary = (
    df_lbe_visits
    .groupBy("CDE_ID")
    .agg(
        F.count("*").alias("P12M_LBE_Visits"),
        F.countDistinct("LocationId").alias("P12M_Distinct_LBE_Locations"),
        F.max("Visit_Date").alias("Last_LBE_Visit_Date"),
        F.min("Visit_Date").alias("First_LBE_Visit_Date")
    )
    .withColumn("Days_Since_Last_LBE", F.datediff(F.lit("2026-07-01"), F.col("Last_LBE_Visit_Date")))
    .withColumn("Visit_Frequency_Bucket",
        F.when(F.col("P12M_LBE_Visits") >= 6, "6+ visits (Habitual)")
         .when(F.col("P12M_LBE_Visits") >= 3, "3-5 visits (Regular)")
         .when(F.col("P12M_LBE_Visits") >= 1, "1-2 visits (Occasional)")
    )
)

# Join back to full eligible population to identify new vs existing
df_segmented = (
    eligible_ids.alias("E")
    .join(df_member_lbe_summary.alias("S"), F.col("E.CDE_ID") == F.col("S.CDE_ID"), "left")
    .withColumn("LBE_Segment",
        F.when(F.col("S.P12M_LBE_Visits").isNull(), "New to LBE")
         .otherwise("Existing LBE Visitor")
    )
    .select(
        F.col("E.CDE_ID"),
        F.coalesce(F.col("S.P12M_LBE_Visits"), F.lit(0)).alias("P12M_LBE_Visits"),
        F.col("S.P12M_Distinct_LBE_Locations"),
        F.col("S.Last_LBE_Visit_Date"),
        F.col("S.Days_Since_Last_LBE"),
        F.col("S.Visit_Frequency_Bucket"),
        F.col("LBE_Segment")
    )
)

# Summary: segment breakdown
df_segment_summary = (
    df_segmented
    .groupBy("LBE_Segment", "Visit_Frequency_Bucket")
    .agg(
        F.count("*").alias("Members"),
        F.round(F.avg("P12M_LBE_Visits"), 1).alias("Avg_Visits")
    )
    .orderBy("LBE_Segment", "Visit_Frequency_Bucket")
)



In [0]:
display(df_segment_summary)

In [0]:
# ── High-level split: what % were already LBE visitors? ──────────────────────
# Avoid unbounded window; compute total once via crossJoin for efficiency
df_counts = df_segmented.groupBy("LBE_Segment").agg(F.count("*").alias("Members"))
total_members = df_counts.agg(F.sum("Members").alias("total"))

df_high_level = (
    df_counts
    .crossJoin(F.broadcast(total_members))
    .withColumn("Pct", F.round(F.col("Members") / F.col("total") * 100, 1))
    .drop("total")
    .orderBy("LBE_Segment")
)



In [0]:
display(df_high_level)

In [0]:
# ── Recency distribution: how recently did existing visitors last go to LBE? ──
_existing_visitors = df_segmented.filter(F.col("LBE_Segment") == "Existing LBE Visitor")

_recency_counts = (
    _existing_visitors
    .withColumn("Recency_Bucket",
        F.when(F.col("Days_Since_Last_LBE") <= 30, "Within 30 days")
         .when(F.col("Days_Since_Last_LBE") <= 90, "31-90 days")
         .when(F.col("Days_Since_Last_LBE") <= 180, "91-180 days")
         .otherwise("180+ days")
    )
    .groupBy("Recency_Bucket")
    .agg(F.count("*").alias("Members"))
)

_total_existing = _recency_counts.agg(F.sum("Members").alias("total"))

df_recency = (
    _recency_counts
    .crossJoin(F.broadcast(_total_existing))
    .withColumn("Pct", F.round(F.col("Members") / F.col("total") * 100, 1))
    .drop("total")
    .orderBy(
        F.when(F.col("Recency_Bucket") == "Within 30 days", 1)
         .when(F.col("Recency_Bucket") == "31-90 days", 2)
         .when(F.col("Recency_Bucket") == "91-180 days", 3)
         .otherwise(4)
    )
)



In [0]:
display(df_recency)

In [0]:
# ── Revenue & Transaction Metrics by Segment (First LBE Visit Post-Minions) ──
T_item = spark.table("cpx_dataanalytics_gold.customer360.cpx_d_item")

# Category definitions (from F&B Trend notebook)
games_categories = {
    "TRR - AMUSEMENT - G&E",
    "TRR - AMUSEMENT - GAMES",
    "TRR - AMUSEMENT - PHYSICAL GAMES",
    "TRR - PROMO - AMUSEMENT",
}
food_categories = {
    "PDM - MENU FOOD - CANDY",
    "TRR - INV ALC BEV - BOTTLE",
    "TRR - INV ALC BEV - CAN",
    "TRR - INV ALC BEV - WINE",
    "TRR - MENU ALC BEV - COCKTAIL",
    "TRR - MENU ALC BEV - DRAUGHT BEER",
    "TRR - MENU BEV - NA COLD BEV",
    "TRR - MENU BEV - NA HOT BEV",
    "TRR - MENU FOOD - APP",
    "TRR - MENU FOOD - DESSERT",
    "TRR - MENU FOOD - KIDS",
    "TRR - MENU FOOD - PIZZA",
    "TRR - MENU FOOD - POUTINE",
    "TRR - MENU FOOD - SANDWICHES",
    "TRR - MOD MENU - ADD-ON",
    "TRR - INV BEV - NA BEV",
    "TRR - MENU ALC BEV - LIQUOR",
    "TRR - MENU ALC BEV - WINE",
    "TRR - MENU FOOD - ENTRÉE",
    "TRR - MENU FOOD - SALAD",
    "VHO-LICENSED FOOD",
    "TRR - INV ALC BEV - LIQUOR",
    "VHO-CANDY",
    "VIP Food - Addons",
    "VHO-LICENSED DRINKS",
    "TRR - MENU FOOD - BRUNCH",
    "VHO-DRINK",
}

# 1. First LBE visit date per customer (earliest visit within 3-week window)
first_visit = F.broadcast(
    df_cross_visit
    .groupBy("CDE_ID")
    .agg(F.min("LBE_Visit_Date").alias("First_LBE_Visit"))
)

# 2. All item-level transactions at LBE locations on the first visit date
df_first_visit_txns = (
    transaction.alias("T")
    .filter(
        F.col("T.CDE_ID").isNotNull()
        & F.col("T.VisitDateId").between("20260701", "20260821")
    )
    .join(lbe_locations.alias("L"), F.col("L.LocationID") == F.col("T.LocationId"), "inner")
    .join(date_dim.alias("D"), F.col("D.dateid") == F.col("T.VisitDateId"), "inner")
    .join(T_item.alias("I"), F.col("I.itemskey") == F.col("T.ItemSkey"), "inner")
    .join(
        first_visit.alias("FV"),
        (F.col("FV.CDE_ID") == F.col("T.CDE_ID"))
        & (F.col("D.datefull") == F.col("FV.First_LBE_Visit")),
        "inner"
    )
    .select(
        F.col("T.CDE_ID"),
        F.col("T.Transaction_Number"),
        F.col("T.SCENEFLAG"),
        F.col("T.Net_Revenue"),
        F.col("I.Item_Class_Description1")
    )
)

# 3. Categorize items & join with segmentation
df_metrics_segmented = (
    df_first_visit_txns.alias("M")
    .withColumn("category",
        F.when(F.col("Item_Class_Description1").isin(list(games_categories)), "Gaming")
         .when(F.col("Item_Class_Description1").isin(list(food_categories)), "F&B")
         .otherwise("Other")
    )
    .join(df_segmented.alias("SEG"), F.col("M.CDE_ID") == F.col("SEG.CDE_ID"), "inner")
)

# 4. Compute metrics per segment
def _agg_metrics(df, label):
    return df.select(
        F.lit(label).alias("Segment"),
        F.countDistinct(F.col("M.CDE_ID")).alias("Customers"),
        F.round(F.sum(F.when(F.col("category") == "Gaming", F.col("Net_Revenue"))), 2).alias("Total_Gaming_Revenue"),
        F.round(F.sum(F.when(F.col("category") == "F&B", F.col("Net_Revenue"))), 2).alias("Total_F&B_Revenue"),
        F.countDistinct("Transaction_Number").alias("Total_Transactions"),
        F.countDistinct(F.when(F.col("SCENEFLAG") == "Yes", F.col("Transaction_Number"))).alias("Total_Scene_Transactions")
    )

df_summary_metrics = (
    _agg_metrics(df_metrics_segmented, "Total")
    .unionByName(_agg_metrics(df_metrics_segmented.filter(F.col("LBE_Segment") == "Existing LBE Visitor"), "Existing Members"))
    .unionByName(_agg_metrics(df_metrics_segmented.filter(F.col("LBE_Segment") == "New to LBE"), "New Members"))
)

display(df_summary_metrics)

In [0]:
# ── Revenue & Transaction Metrics by Segment × LBE_Brand (First LBE Visit Post-Minions) ──
T_item = spark.table("cpx_dataanalytics_gold.customer360.cpx_d_item")

games_categories = {
    "TRR - AMUSEMENT - G&E",
    "TRR - AMUSEMENT - GAMES",
    "TRR - AMUSEMENT - PHYSICAL GAMES",
    "TRR - PROMO - AMUSEMENT",
}
food_categories = {
    "PDM - MENU FOOD - CANDY",
    "TRR - INV ALC BEV - BOTTLE", "TRR - INV ALC BEV - CAN", "TRR - INV ALC BEV - WINE",
    "TRR - MENU ALC BEV - COCKTAIL", "TRR - MENU ALC BEV - DRAUGHT BEER",
    "TRR - MENU BEV - NA COLD BEV", "TRR - MENU BEV - NA HOT BEV",
    "TRR - MENU FOOD - APP", "TRR - MENU FOOD - DESSERT", "TRR - MENU FOOD - KIDS",
    "TRR - MENU FOOD - PIZZA", "TRR - MENU FOOD - POUTINE", "TRR - MENU FOOD - SANDWICHES",
    "TRR - MOD MENU - ADD-ON", "TRR - INV BEV - NA BEV",
    "TRR - MENU ALC BEV - LIQUOR", "TRR - MENU ALC BEV - WINE",
    "TRR - MENU FOOD - ENTRÉE", "TRR - MENU FOOD - SALAD",
    "VHO-LICENSED FOOD", "TRR - INV ALC BEV - LIQUOR", "VHO-CANDY",
    "VIP Food - Addons", "VHO-LICENSED DRINKS", "TRR - MENU FOOD - BRUNCH", "VHO-DRINK",
}

# 1. First LBE visit date per customer
first_visit = F.broadcast(
    df_cross_visit
    .groupBy("CDE_ID")
    .agg(F.min("LBE_Visit_Date").alias("First_LBE_Visit"))
)

# 2. Item-level txns on first-visit date — now carries LBE_Brand from location table
df_first_visit_txns = (
    transaction.alias("T")
    .filter(
        F.col("T.CDE_ID").isNotNull()
        & F.col("T.VisitDateId").between("20260701", "20260821")
    )
    .join(lbe_locations.alias("L"), F.col("L.LocationID") == F.col("T.LocationId"), "inner")
    .join(date_dim.alias("D"), F.col("D.dateid") == F.col("T.VisitDateId"), "inner")
    .join(T_item.alias("I"), F.col("I.itemskey") == F.col("T.ItemSkey"), "inner")
    .join(
        first_visit.alias("FV"),
        (F.col("FV.CDE_ID") == F.col("T.CDE_ID"))
        & (F.col("D.datefull") == F.col("FV.First_LBE_Visit")),
        "inner"
    )
    .select(
        F.col("T.CDE_ID"),
        F.col("T.Transaction_Number"),
        F.col("T.SCENEFLAG"),
        F.col("T.Net_Revenue"),
        F.col("I.Item_Class_Description1"),
        F.col("L.LC_Trade_Brand_Description").alias("LBE_Brand"),
    )
)

# 3. Categorize + join segmentation
df_metrics_segmented = (
    df_first_visit_txns.alias("M")
    .withColumn("category",
        F.when(F.col("Item_Class_Description1").isin(list(games_categories)), "Gaming")
         .when(F.col("Item_Class_Description1").isin(list(food_categories)), "F&B")
         .otherwise("Other")
    )
    .join(df_segmented.alias("SEG"), F.col("M.CDE_ID") == F.col("SEG.CDE_ID"), "inner")
)

# 4. Metrics per segment × brand
def _agg_metrics(df, label):
    return (
        df.groupBy("LBE_Brand")
        .agg(
            F.countDistinct(F.col("M.CDE_ID")).alias("Customers"),
            F.round(F.sum(F.when(F.col("category") == "Gaming", F.col("Net_Revenue"))), 2).alias("Total_Gaming_Revenue"),
            F.round(F.sum(F.when(F.col("category") == "F&B", F.col("Net_Revenue"))), 2).alias("Total_F&B_Revenue"),
            F.countDistinct("Transaction_Number").alias("Total_Transactions"),
            F.countDistinct(F.when(F.col("SCENEFLAG") == "Yes", F.col("Transaction_Number"))).alias("Total_Scene_Transactions"),
        )
        .withColumn("Segment", F.lit(label))
        .select("Segment", "LBE_Brand", "Customers",
                "Total_Gaming_Revenue", "Total_F&B_Revenue",
                "Total_Transactions", "Total_Scene_Transactions")
    )

df_summary_metrics = (
    _agg_metrics(df_metrics_segmented, "Total")
    .unionByName(_agg_metrics(df_metrics_segmented.filter(F.col("LBE_Segment") == "Existing LBE Visitor"), "Existing Members"))
    .unionByName(_agg_metrics(df_metrics_segmented.filter(F.col("LBE_Segment") == "New to LBE"), "New Members"))
    .orderBy("Segment", "LBE_Brand")
)

display(df_summary_metrics)

In [0]:
# ── Age group breakdown of qualifying members ──
customer = spark.table("cpx_dataanalytics_gold.customer360.cpx_d_customer")

# Small member list from cross-visit table (one row per member per brand visited)
qual_members = F.broadcast(
    df_cross_visit
    .select("CDE_ID", "LBE_Brand")
    .distinct()
)

seg_bcast = F.broadcast(df_segmented.select("CDE_ID", "LBE_Segment"))

# If date_of_birth_masked is a DATE, use F.year(...). If it's already a year int, use it directly.
# Adjust the age_expr line accordingly.
age_expr = F.year(F.current_date()) - (F.col("date_of_birth_masked"))
# age_expr = F.lit(2026) - F.col("date_of_birth_masked")   # <- use this if column is INT year

df_age = (
    qual_members.alias("Q")
    .join(seg_bcast.alias("S"), "CDE_ID", "left")
    .join(
        customer.filter(F.col("customer_status") == "Active")
                .select("cde_id", "date_of_birth_masked").alias("C"),
        F.col("C.cde_id") == F.col("Q.CDE_ID"),
        "left"
    )
    .withColumn("Age", age_expr)
    .withColumn(
        "Age_Group",
        F.when(F.col("Age").isNull(), "Unknown")
         .when(F.col("Age") < 18, "Under 18")
         .when(F.col("Age") < 25, "18-24")
         .when(F.col("Age") < 35, "25-34")
         .when(F.col("Age") < 45, "35-44")
         .when(F.col("Age") < 55, "45-54")
         .when(F.col("Age") < 65, "55-64")
         .otherwise("65+")
    )
    .select(F.col("Q.CDE_ID").alias("CDE_ID"), "LBE_Segment", "LBE_Brand", "Age_Group")
)

df_age_overall = (
    df_age.groupBy("Age_Group")
    .agg(F.countDistinct("CDE_ID").alias("Members"))
    .orderBy("Age_Group")
)
display(df_age_overall)

df_age_seg_brand = (
    df_age.groupBy("LBE_Segment", "LBE_Brand", "Age_Group")
    .agg(F.countDistinct("CDE_ID").alias("Members"))
    .orderBy("LBE_Segment", "LBE_Brand", "Age_Group")
)
display(df_age_seg_brand)

In [0]:
# ── Days between Minions purchase and LBE visit ──
df_days_diff = (
    df_cross_visit
    .withColumn("Days_To_LBE", F.datediff("LBE_Visit_Date", "Minions_Purchase_Date"))
)

# Distribution
df_days_dist = (
    df_days_diff
    .groupBy("Days_To_LBE")
    .agg(
        F.countDistinct("CDE_ID").alias("Members"),
        F.count("*").alias("Visits")
    )
    .orderBy("Days_To_LBE")
)
display(df_days_dist)

# Bucketed view
df_days_buckets = (
    df_days_diff
    .withColumn(
        "Days_Bucket",
        F.when(F.col("Days_To_LBE") == 0, "Same day")
         .when(F.col("Days_To_LBE") <= 3, "1-3 days")
         .when(F.col("Days_To_LBE") <= 7, "4-7 days")
         .when(F.col("Days_To_LBE") <= 14, "8-14 days")
         .otherwise("15-21 days")
    )
    .groupBy("Days_Bucket")
    .agg(
        F.countDistinct("CDE_ID").alias("Members"),
        F.count("*").alias("Visits")
    )
    .orderBy(
        F.when(F.col("Days_Bucket") == "Same day", 1)
         .when(F.col("Days_Bucket") == "1-3 days", 2)
         .when(F.col("Days_Bucket") == "4-7 days", 3)
         .when(F.col("Days_Bucket") == "8-14 days", 4)
         .otherwise(5)
    )
)
display(df_days_buckets)

# Summary stats
df_days_summary = df_days_diff.agg(
    F.round(F.avg("Days_To_LBE"), 1).alias("Avg_Days"),
    F.expr("percentile_approx(Days_To_LBE, 0.5)").alias("Median_Days"),
    F.min("Days_To_LBE").alias("Min_Days"),
    F.max("Days_To_LBE").alias("Max_Days")
)
display(df_days_summary)

In [0]:
# ── Revenue & Transaction Metrics by Segment × LBE_Brand ──
T_item = spark.table("cpx_dataanalytics_gold.customer360.cpx_d_item")

games_categories = [
    "TRR - AMUSEMENT - G&E",
    "TRR - AMUSEMENT - GAMES",
    "TRR - AMUSEMENT - PHYSICAL GAMES",
    "TRR - PROMO - AMUSEMENT",
]
food_categories = [
    "PDM - MENU FOOD - CANDY",
    "TRR - INV ALC BEV - BOTTLE", "TRR - INV ALC BEV - CAN", "TRR - INV ALC BEV - WINE",
    "TRR - MENU ALC BEV - COCKTAIL", "TRR - MENU ALC BEV - DRAUGHT BEER",
    "TRR - MENU BEV - NA COLD BEV", "TRR - MENU BEV - NA HOT BEV",
    "TRR - MENU FOOD - APP", "TRR - MENU FOOD - DESSERT", "TRR - MENU FOOD - KIDS",
    "TRR - MENU FOOD - PIZZA", "TRR - MENU FOOD - POUTINE", "TRR - MENU FOOD - SANDWICHES",
    "TRR - MOD MENU - ADD-ON", "TRR - INV BEV - NA BEV",
    "TRR - MENU ALC BEV - LIQUOR", "TRR - MENU ALC BEV - WINE",
    "TRR - MENU FOOD - ENTRÉE", "TRR - MENU FOOD - SALAD",
    "VHO-LICENSED FOOD", "TRR - INV ALC BEV - LIQUOR", "VHO-CANDY",
    "VIP Food - Addons", "VHO-LICENSED DRINKS", "TRR - MENU FOOD - BRUNCH", "VHO-DRINK",
]

# 1. Qualifying (member × date) pairs as yyyyMMdd int — small, broadcastable
qualifying_visits = F.broadcast(
    df_cross_visit
    .groupBy("CDE_ID", "Minions_Purchase_Date")
    .agg(F.min("LBE_Visit_Date").alias("Qualifying_LBE_Visit"))
    .select(
        "CDE_ID",
        F.date_format("Qualifying_LBE_Visit", "yyyyMMdd").cast("int").alias("Qualifying_DateId"),
    )
    .distinct()
)

# 2. Broadcast segmentation (~1.5k rows)
seg_bcast = F.broadcast(df_segmented.select("CDE_ID", "LBE_Segment"))

# 3. Item-level txns on qualifying dates — join on VisitDateId directly (no date_dim)
df_qual_visit_txns = (
    transaction.alias("T")
    .filter(
        F.col("T.CDE_ID").isNotNull()
        & F.col("T.VisitDateId").between(20260701, 20260821)
    )
    .join(
        qualifying_visits.alias("QV"),
        (F.col("QV.CDE_ID") == F.col("T.CDE_ID"))
        & (F.col("QV.Qualifying_DateId") == F.col("T.VisitDateId")),
        "inner"
    )
    .join(lbe_locations.alias("L"), F.col("L.LocationID") == F.col("T.LocationId"), "inner")
    .join(T_item.alias("I"), F.col("I.itemskey") == F.col("T.ItemSkey"), "inner")
    .join(seg_bcast.alias("SEG"), F.col("SEG.CDE_ID") == F.col("T.CDE_ID"), "inner")
    .select(
        F.col("T.CDE_ID"),
        F.col("T.Transaction_Number"),
        F.col("T.SCENEFLAG"),
        F.col("T.Net_Revenue"),
        F.col("L.LC_Trade_Brand_Description").alias("LBE_Brand"),
        F.col("SEG.LBE_Segment"),
        F.when(F.col("I.Item_Class_Description1").isin(games_categories), "Gaming")
         .when(F.col("I.Item_Class_Description1").isin(food_categories), "F&B")
         .otherwise("Other").alias("category"),
    )
)

# 4. Single-pass aggregation: Segment × Brand
def _agg(df, seg_label):
    return (
        df.groupBy("LBE_Brand")
        .agg(
            F.countDistinct("CDE_ID").alias("Customers"),
            F.round(F.sum(F.when(F.col("category") == "Gaming", F.col("Net_Revenue"))), 2).alias("Total_Gaming_Revenue"),
            F.round(F.sum(F.when(F.col("category") == "F&B",    F.col("Net_Revenue"))), 2).alias("Total_F&B_Revenue"),
            F.countDistinct("Transaction_Number").alias("Total_Transactions"),
            F.countDistinct(F.when(F.col("SCENEFLAG") == "Yes", F.col("Transaction_Number"))).alias("Total_Scene_Transactions"),
        )
        .withColumn("Segment", F.lit(seg_label))
        .select("Segment", "LBE_Brand", "Customers",
                "Total_Gaming_Revenue", "Total_F&B_Revenue",
                "Total_Transactions", "Total_Scene_Transactions")
    )

# Cache the built dataset — 3 aggregations feed off it
df_qual_visit_txns = df_qual_visit_txns.cache()

df_summary_metrics = (
    _agg(df_qual_visit_txns, "Total")
    .unionByName(_agg(df_qual_visit_txns.filter(F.col("LBE_Segment") == "Existing LBE Visitor"), "Existing Members"))
    .unionByName(_agg(df_qual_visit_txns.filter(F.col("LBE_Segment") == "New to LBE"), "New Members"))
    .orderBy("Segment", "LBE_Brand")
)

display(df_summary_metrics)